# Move Operator Comparison

This notebook compares the three basic move operators for Dense Graph Partition experiments across Powerlaw and Erdős-Rényi instances:

- `move_first`
- `move_best`
- `move_plateau`

The analysis is grouped by graph type, size class, and density regime.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Configuration

The following configuration defines all result directories that should be included in the analysis.

In [4]:
MOVE_OPERATORS = ["move_first", "move_best", "move_plateau"]

EXPERIMENTS = [
    {
        "graph_type": "powerlaw",
        "dataset": "small sparse",
        "results_dir": Path(
            "../results/experiment2_analysis/move_operator/"
            "move_operator_small_powerlaw_sparse"
        ),
    },
    {
        "graph_type": "powerlaw",
        "dataset": "small dense",
        "results_dir": Path(
            "../results/experiment2_analysis/move_operator/"
            "move_operator_small_powerlaw_dense"
        ),
    },
    {
        "graph_type": "powerlaw",
        "dataset": "large sparse",
        "results_dir": Path(
            "../results/experiment2_analysis/move_operator/"
            "move_operator_large_powerlaw_sparse"
        ),
    },
    {
        "graph_type": "powerlaw",
        "dataset": "large dense",
        "results_dir": Path(
            "../results/experiment2_analysis/move_operator/"
            "move_operator_large_powerlaw_dense"
        ),
    },
    {
        "graph_type": "er",
        "dataset": "small sparse",
        "results_dir": Path(
            "../results/experiment2_analysis/move_operator/"
            "move_operator_small_er_sparse"
        ),
    },
    {
        "graph_type": "er",
        "dataset": "small dense",
        "results_dir": Path(
            "../results/experiment2_analysis/move_operator/"
            "move_operator_small_er_dense"
        ),
    },
    {
        "graph_type": "er",
        "dataset": "large sparse",
        "results_dir": Path(
            "../results/experiment2_analysis/move_operator/"
            "move_operator_large_er_sparse"
        ),
    },
    {
        "graph_type": "er",
        "dataset": "large dense",
        "results_dir": Path(
            "../results/experiment2_analysis/move_operator/"
            "move_operator_large_er_dense"
        ),
    },
]

## Load experiment results

All raw result tables are loaded and annotated with their graph type and dataset group.

In [5]:
raw_rows = []

for experiment in EXPERIMENTS:
    raw = pd.read_csv(experiment["results_dir"] / "raw_results.csv")

    raw["graph_type"] = experiment["graph_type"]
    raw["dataset_group"] = experiment["dataset"]

    raw_rows.append(raw)

raw_all = pd.concat(raw_rows, ignore_index=True)

raw_all.shape

(420000, 34)

## Select move-only pipelines

Only experiments consisting of one of the three basic move operators are considered.

In [6]:
move_raw = raw_all[raw_all["pipeline"].isin(MOVE_OPERATORS)].copy()

move_raw["pipeline"] = pd.Categorical(
    move_raw["pipeline"],
    categories=MOVE_OPERATORS,
    ordered=True,
)

move_raw.shape

(420000, 34)

## Select the best run per instance and operator

For solution quality, repeated runs are not treated as separate instances. For each graph type, dataset group, instance, and operator, only the run with the highest final density is kept.

In [7]:
best_runs = (
    move_raw
    .sort_values("final_density", ascending=False)
    .groupby(
        ["graph_type", "dataset_group", "dataset", "instance", "pipeline"],
        observed=True,
        as_index=False,
    )
    .head(1)
    .reset_index(drop=True)
)

best_runs.shape

(6000, 34)

## Build instance-level density table

The density table contains one row per instance and one column per move operator.


In [8]:
density_table = best_runs.pivot_table(
    index=["graph_type", "dataset_group", "dataset", "instance"],
    columns="pipeline",
    values="final_density",
    observed=True,
)

density_table = density_table[MOVE_OPERATORS]

density_table.head()

pipeline                                                                move_first  \
graph_type dataset_group dataset        instance                                     
er         large dense   er_dense_large er_dense_large_000_n501_s71126    160.6500   
                                        er_dense_large_001_n505_s71234    162.1000   
                                        er_dense_large_002_n506_s71138    164.1167   
                                        er_dense_large_003_n506_s71206    163.2500   
                                        er_dense_large_004_n518_s71082    164.5167   

pipeline                                                                move_best  \
graph_type dataset_group dataset        instance                                    
er         large dense   er_dense_large er_dense_large_000_n501_s71126   160.6500   
                                        er_dense_large_001_n505_s71234   162.1000   
                                        er_dense_large_002_n506_s71138   164.1167   
                                        er_dense_large_003_n506_s71206   163.2500   
                                        er_dense_large_004_n518_s71082   164.5167   

pipeline                                                                move_plateau  
graph_type dataset_group dataset        instance                                      
er         large dense   er_dense_large er_dense_large_000_n501_s71126        160.65  
                                        er_dense_large_001_n505_s71234        162.10  
                                        er_dense_large_002_n506_s71138        164.35  
                                        er_dense_large_003_n506_s71206        163.25  
                                        er_dense_large_004_n518_s71082        164.60

## Relative solution quality

Each operator is compared to the best result found by any move operator on the same instance.


In [9]:
best_per_instance = density_table.max(axis=1)
relative_to_best = density_table.div(best_per_instance, axis=0)

relative_to_best_summary = (
    relative_to_best
    .groupby(level=["graph_type", "dataset_group"])
    .agg(["mean", "median", "min", "max"])
)

relative_to_best_summary = (
    relative_to_best_summary
    .stack(level=0, future_stack=True)
    .reset_index()
)

relative_to_best_summary = relative_to_best_summary.rename(
    columns={
        "pipeline": "operator",
        "level_2": "operator",
    }
)

relative_to_best_summary

,graph_type,dataset_group,operator,mean,median,min,max
0,er,large dense,move_first,0.999692,0.999890,0.997977,1.000000
1,er,large dense,move_best,0.999681,0.999871,0.997977,1.000000
2,er,large dense,move_plateau,0.999999,1.000000,0.999778,1.000000
3,er,large sparse,move_first,0.998760,0.999021,0.989622,1.000000
4,er,large sparse,move_best,0.998760,0.999021,0.989622,1.000000
5,er,large sparse,move_plateau,0.999999,1.000000,0.999645,1.000000
6,er,small dense,move_first,0.995102,0.998352,0.963492,1.000000
7,er,small dense,move_best,0.995090,0.998254,0.963492,1.000000
8,er,small dense,move_plateau,0.999999,1.000000,0.999734,1.000000
9,er,small sparse,move_first,0.994464,0.997205,0.946429,1.000000


## Winner counts

An operator is counted as best if it reaches the best value on an instance. Ties are counted for all operators involved.


In [10]:
is_best = density_table.eq(best_per_instance, axis=0)
num_best = is_best.sum(axis=1)

best_counts = (
    is_best
    .groupby(level=["graph_type", "dataset_group"])
    .sum()
)

tied_best_counts = (
    is_best
    .where(num_best > 1, False)
    .groupby(level=["graph_type", "dataset_group"])
    .sum()
)

winner_counts = (
    best_counts
    .stack()
    .rename("best_count_including_ties")
    .reset_index()
    .rename(columns={"pipeline": "operator"})
)

winner_counts["tied_best_count"] = tied_best_counts.stack().values

winner_counts["unique_best_count"] = (
        winner_counts["best_count_including_ties"]
        - winner_counts["tied_best_count"]
)

num_instances = (
    density_table
    .groupby(level=["graph_type", "dataset_group"])
    .size()
    .rename("num_instances")
    .reset_index()
)

winner_counts = winner_counts.merge(
    num_instances,
    on=["graph_type", "dataset_group"],
)

winner_counts["winner_rate"] = (
        winner_counts["best_count_including_ties"]
        / winner_counts["num_instances"]
)

winner_counts

,graph_type,dataset_group,operator,best_count_including_ties,tied_best_count,unique_best_count,num_instances,winner_rate
0,er,large dense,move_first,120,120,0,250,0.480
1,er,large dense,move_best,118,118,0,250,0.472
2,er,large dense,move_plateau,249,119,130,250,0.996
3,er,large sparse,move_first,80,80,0,250,0.320
4,er,large sparse,move_best,80,80,0,250,0.320
5,er,large sparse,move_plateau,249,79,170,250,0.996
6,er,small dense,move_first,107,107,0,250,0.428
7,er,small dense,move_best,107,107,0,250,0.428
8,er,small dense,move_plateau,249,108,141,250,0.996
9,er,small sparse,move_first,88,88,0,250,0.352


## Runtime analysis

The solution quality is evaluated using the best result out of repeated runs. To ensure a fair runtime comparison, the reported runtime corresponds to the total runtime of all runs for each instance and operator.


In [11]:
runtime_per_instance = (
    move_raw
    .groupby(
        ["graph_type", "dataset_group", "dataset", "instance", "pipeline"],
        observed=True,
    )
    .agg(
        total_runtime=("total_runtime", "sum"),
        mean_num_moves=("num_moves", "mean"),
        mean_num_passes=("num_passes", "mean"),
    )
    .reset_index()
)

runtime_summary = (
    runtime_per_instance
    .groupby(
        ["graph_type", "dataset_group", "pipeline"],
        observed=True,
    )
    .agg(
        instances=("instance", "count"),
        mean_runtime=("total_runtime", "mean"),
        median_runtime=("total_runtime", "median"),
        mean_num_moves=("mean_num_moves", "mean"),
        mean_num_passes=("mean_num_passes", "mean"),
    )
    .reset_index()
    .rename(columns={"pipeline": "operator"})
)

runtime_summary

,graph_type,dataset_group,operator,instances,mean_runtime,median_runtime,mean_num_moves,mean_num_passes
0,er,large dense,move_first,250,164.977216,124.98600,94.767086,95.767086
1,er,large dense,move_best,250,4178.500576,2486.38975,94.126000,95.126000
2,er,large dense,move_plateau,250,591.613822,387.92135,2084.151086,15.538629
3,er,large sparse,move_first,250,22.130151,20.67630,124.357600,125.357600
4,er,large sparse,move_best,250,320.117710,307.33335,123.027429,124.027429
5,er,large sparse,move_plateau,250,187.119320,173.60685,2150.427943,77.019943
6,er,small dense,move_first,250,30.976519,31.25380,21.114057,22.114057
7,er,small dense,move_best,250,130.210428,119.21600,20.579429,21.579429
8,er,small dense,move_plateau,250,143.349202,137.42880,355.807371,25.069771
9,er,small sparse,move_first,250,21.541131,21.53530,24.159371,25.159371


## Combined comparison table

In [12]:
main_comparison_all = (
    relative_to_best_summary
    .rename(
        columns={
            "mean": "mean_relative_to_best",
            "median": "median_relative_to_best",
            "min": "min_relative_to_best",
            "max": "max_relative_to_best",
        }
    )
    .merge(
        winner_counts.drop(columns=["num_instances"]),
        on=["graph_type", "dataset_group", "operator"],
    )
    .merge(
        runtime_summary,
        on=["graph_type", "dataset_group", "operator"],
    )
)

main_comparison_all

,graph_type,dataset_group,operator,mean_relative_to_best,median_relative_to_best,min_relative_to_best,max_relative_to_best,best_count_including_ties,tied_best_count,unique_best_count,winner_rate,instances,mean_runtime,median_runtime,mean_num_moves,mean_num_passes
0,er,large dense,move_first,0.999692,0.999890,0.997977,1.000000,120,120,0,0.480,250,164.977216,124.98600,94.767086,95.767086
1,er,large dense,move_best,0.999681,0.999871,0.997977,1.000000,118,118,0,0.472,250,4178.500576,2486.38975,94.126000,95.126000
2,er,large dense,move_plateau,0.999999,1.000000,0.999778,1.000000,249,119,130,0.996,250,591.613822,387.92135,2084.151086,15.538629
3,er,large sparse,move_first,0.998760,0.999021,0.989622,1.000000,80,80,0,0.320,250,22.130151,20.67630,124.357600,125.357600
4,er,large sparse,move_best,0.998760,0.999021,0.989622,1.000000,80,80,0,0.320,250,320.117710,307.33335,123.027429,124.027429
5,er,large sparse,move_plateau,0.999999,1.000000,0.999645,1.000000,249,79,170,0.996,250,187.119320,173.60685,2150.427943,77.019943
6,er,small dense,move_first,0.995102,0.998352,0.963492,1.000000,107,107,0,0.428,250,30.976519,31.25380,21.114057,22.114057
7,er,small dense,move_best,0.995090,0.998254,0.963492,1.000000,107,107,0,0.428,250,130.210428,119.21600,20.579429,21.579429
8,er,small dense,move_plateau,0.999999,1.000000,0.999734,1.000000,249,108,141,0.996,250,143.349202,137.42880,355.807371,25.069771
9,er,small sparse,move_first,0.994464,0.997205,0.946429,1.000000,88,88,0,0.352,250,21.541131,21.53530,24.159371,25.159371


In [13]:
EXPORT_DIR = Path(
    "../results/experiment2_analysis/move_operator_combined_analysis"
)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# main_comparison_all.to_csv(
#     EXPORT_DIR / "main_comparison_all.csv",
#     index=False,
# )
# relative_to_best_summary.to_csv(
#     EXPORT_DIR / "relative_to_best_all.csv",
#     index=False,
# )
# winner_counts.to_csv(
#     EXPORT_DIR / "winner_counts_all.csv",
#     index=False,
# )
# runtime_summary.to_csv(
#     EXPORT_DIR / "runtime_summary_all.csv",
#     index=False,
# )

EXPORT_DIR

PosixPath('../results/experiment2_analysis/move_operator_combined_analysis')

## LaTeX helper functions

In [14]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_operator(operator: str) -> str:
    return r"\texttt{" + operator.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"


def format_percent(value: float, decimals: int = 1) -> str:
    return (
        f"{truncate_number(100 * value, decimals):.{decimals}f}"
        r"\%"
    )

## Build runtime LaTeX table

In [15]:
def make_runtime_latex_table(
        df: pd.DataFrame,
        caption: str,
        label: str,
) -> str:
    graph_order = ["powerlaw", "er"]
    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": r"Erdős--Rényi",
    }
    dataset_order = [
        "small sparse",
        "small dense",
        "large sparse",
        "large dense",
    ]

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lllrrr}",
        r"\toprule",
        (
            r"Graphentyp & Datensatz & Operator & Laufzeit (s) "
            r"& Knotenverschiebungen & Durchläufe \\"
        ),
        r"\midrule",
    ]

    for graph_type in graph_order:
        graph_df = df[df["graph_type"] == graph_type]

        for dataset_index, dataset in enumerate(dataset_order):
            part = graph_df[
                graph_df["dataset_group"] == dataset
                ].copy()

            part["operator"] = pd.Categorical(
                part["operator"],
                categories=MOVE_OPERATORS,
                ordered=True,
            )
            part = part.sort_values("operator")

            if part.empty:
                continue

            for i, row in enumerate(part.itertuples(index=False)):
                graph_cell = (
                    rf"\multirow{{12}}{{*}}"
                    rf"{{{graph_labels[graph_type]}}}"
                    if dataset_index == 0 and i == 0
                    else ""
                )
                dataset_cell = (
                    rf"\multirow{{3}}{{*}}{{{dataset}}}"
                    if i == 0
                    else ""
                )

                lines.append(
                    f"{graph_cell} & {dataset_cell} "
                    f"& {latex_operator(str(row.operator))} "
                    f"& {format_number(row.mean_runtime, 2)} "
                    f"& {format_number(row.mean_num_moves, 1)} "
                    f"& {format_number(row.mean_num_passes, 1)} \\\\"
                )

            if dataset_index < len(dataset_order) - 1:
                lines.append(r"\cmidrule(l){2-6}")
            else:
                lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.extend(
        [
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)


runtime_latex = make_runtime_latex_table(
    main_comparison_all,
    caption=(
        "Mittlere Gesamtlaufzeit der zehn Runs sowie durchschnittliche "
        "Anzahl ausgeführter Knotenverschiebungen und Durchläufe auf "
        "Powerlaw- und Erdős-Rényi-Instanzen."
    ),
    label="tab:move_runtime",
)

print(runtime_latex)

\begin{table}[t]
\centering
\caption{Mittlere Gesamtlaufzeit der zehn Runs sowie durchschnittliche Anzahl ausgeführter Knotenverschiebungen und Durchläufe auf Powerlaw- und Erdős-Rényi-Instanzen.}
\label{tab:move_runtime}
\begin{tabular}{lllrrr}
\toprule
Graphentyp & Datensatz & Operator & Laufzeit (s) & Knotenverschiebungen & Durchläufe \\
\midrule
\multirow{12}{*}{Powerlaw} & \multirow{3}{*}{small sparse} & \texttt{move\_first} & 2.65 & 39.3 & 40.3 \\
 &  & \texttt{move\_best} & 13.59 & 36.0 & 37.0 \\
 &  & \texttt{move\_plateau} & 7.32 & 348.4 & 24.1 \\
\cmidrule(l){2-6}
 & \multirow{3}{*}{small dense} & \texttt{move\_first} & 6.64 & 39.0 & 40.0 \\
 &  & \texttt{move\_best} & 34.73 & 35.7 & 36.7 \\
 &  & \texttt{move\_plateau} & 21.03 & 366.7 & 25.5 \\
\cmidrule(l){2-6}
 & \multirow{3}{*}{large sparse} & \texttt{move\_first} & 23.95 & 202.9 & 203.9 \\
 &  & \texttt{move\_best} & 531.53 & 193.1 & 194.1 \\
 &  & \texttt{move\_plateau} & 61.95 & 2250.3 & 20.9 \\
\cmidrule(l){2-6}
 & \m

## Build quality LaTeX table

In [16]:
def make_quality_latex_table(
        df: pd.DataFrame,
        caption: str,
        label: str,
) -> str:
    graph_order = ["powerlaw", "er"]
    graph_labels = {
        "powerlaw": "Powerlaw",
        "er": r"Erdős--Rényi",
    }
    dataset_order = [
        "small sparse",
        "small dense",
        "large sparse",
        "large dense",
    ]

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabular}{lllrr}",
        r"\toprule",
        (
            r"Graphentyp & Datensatz & Operator "
            r"& Mittelwert & Gewinnrate \\"
        ),
        r"\midrule",
    ]

    for graph_type in graph_order:
        graph_df = df[df["graph_type"] == graph_type]

        if graph_df.empty:
            continue

        for dataset_index, dataset in enumerate(dataset_order):
            part = graph_df[
                graph_df["dataset_group"] == dataset
                ].copy()

            if part.empty:
                continue

            part["operator"] = pd.Categorical(
                part["operator"],
                categories=MOVE_OPERATORS,
                ordered=True,
            )
            part = part.sort_values("operator")

            for i, row in enumerate(part.itertuples(index=False)):
                graph_cell = (
                    rf"\multirow{{12}}{{*}}"
                    rf"{{{graph_labels[graph_type]}}}"
                    if dataset_index == 0 and i == 0
                    else ""
                )
                dataset_cell = (
                    rf"\multirow{{3}}{{*}}{{{dataset}}}"
                    if i == 0
                    else ""
                )

                mean = format_number(
                    row.mean_relative_to_best,
                    4,
                )
                winner_rate = format_percent(
                    row.winner_rate,
                    1,
                )

                if row.operator == "move_plateau":
                    mean = rf"\textbf{{{mean}}}"
                    winner_rate = rf"\textbf{{{winner_rate}}}"

                lines.append(
                    f"{graph_cell} & {dataset_cell} "
                    f"& {latex_operator(str(row.operator))} "
                    f"& {mean} & {winner_rate} \\\\"
                )

            if dataset_index < len(dataset_order) - 1:
                lines.append(r"\cmidrule(l){2-5}")
            else:
                lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.extend(
        [
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)


quality_latex = make_quality_latex_table(
    main_comparison_all,
    caption=(
        "Relative Lösungsqualität der Move-Operatoren. Angegeben ist "
        "der Quotient zur besten auf derselben Instanz gefundenen "
        "Lösung sowie die Gewinnrate inklusive Gleichständen."
    ),
    label="tab:move_quality",
)

print(quality_latex)

\begin{table}[t]
\centering
\caption{Relative Lösungsqualität der Move-Operatoren. Angegeben ist der Quotient zur besten auf derselben Instanz gefundenen Lösung sowie die Gewinnrate inklusive Gleichständen.}
\label{tab:move_quality}
\begin{tabular}{lllrr}
\toprule
Graphentyp & Datensatz & Operator & Mittelwert & Gewinnrate \\
\midrule
\multirow{12}{*}{Powerlaw} & \multirow{3}{*}{small sparse} & \texttt{move\_first} & 0.9839 & 3.2\% \\
 &  & \texttt{move\_best} & 0.9836 & 3.2\% \\
 &  & \texttt{move\_plateau} & \textbf{1.0000} & \textbf{100.0\%} \\
\cmidrule(l){2-5}
 & \multirow{3}{*}{small dense} & \texttt{move\_first} & 0.9846 & 0.4\% \\
 &  & \texttt{move\_best} & 0.9837 & 0.8\% \\
 &  & \texttt{move\_plateau} & \textbf{1.0000} & \textbf{100.0\%} \\
\cmidrule(l){2-5}
 & \multirow{3}{*}{large sparse} & \texttt{move\_first} & 0.9956 & 0.0\% \\
 &  & \texttt{move\_best} & 0.9956 & 0.0\% \\
 &  & \texttt{move\_plateau} & \textbf{1.0000} & \textbf{100.0\%} \\
\cmidrule(l){2-5}
 & \multiro